In [1]:
import numpy as np 
import pandas as pd 
import os
from datasets import load_dataset 
from sklearn.model_selection import GroupShuffleSplit 

#fixed seeds for reproducibility
random_state = 42

In [2]:
# load dataset from hugging face
data = load_dataset("b-mc2/sql-create-context")

#convert the data to a Pandas DataFrame
df = data["train"].to_pandas()

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
#splitting train/test/val without data leakage
#randomly divide data into subsets while keeping entire schema groups together
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state = random_state) #split to training and holdout(test and val)
train_idx, hold_idx = next(gss.split(df, groups=df["context"])) #the context column defines the groups
train_df, hold_df = df.iloc[train_idx], df.iloc[hold_idx] #convert arrays to dataframes 

#split the holdout 20% of groups in 2 : val and test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state = random_state)
val_idx, test_idx = next(gss2.split(hold_df, groups=hold_df["context"]))
val_df, test_df = hold_df.iloc[val_idx], hold_df.iloc[test_idx]

print(len(train_df), len(val_df), len(test_df))

print("Train/Validation schema overlap:",
      len(set(train_df["context"]) & set(val_df["context"])))

print("Train/Test schema overlap:",
      len(set(train_df["context"]) & set(test_df["context"])))

print("Validation/Test schema overlap:",
      len(set(val_df["context"]) & set(test_df["context"])))

62895 7859 7823
Train/Validation schema overlap: 0
Train/Test schema overlap: 0
Validation/Test schema overlap: 0
